## Goal

> ⚠️ **Research on public allocation data.** Not investment advice; results describe the methods, not the funds.

> 🇸🇪 **Categories and cluster names are in Swedish**, as delivered by the source. Common words: *Branschfond* = sector fund, *Sverige* = Sweden, *Norden* = Nordics, *Små-/medelstora bolag* = small/mid caps, *Mix bolag* = mixed, *Råvaror* = commodities, *Läkemedel* = pharma.

What happened to each **cluster** from week to week? `cluster_migration.ipynb` follows funds between two weeks; this notebook follows the clusters themselves through every week in `WEEKS`, Hellinger only.

## How transitions are labelled

Cluster ids are renumbered on every run, so clusters are followed by the **funds they share** (only funds present in both weeks count). A *piece* is at least a quarter of a cluster's funds. This is the MONIC approach (Spiliopoulou et al., 2006).

| Week | Transition | Meaning |
|---|---|---|
| earlier | `survived` | More than half its funds went into one cluster, and they are more than half of it |
| earlier | `split` | Two or more pieces, none a survival |
| earlier | `absorbed` | Most of its funds went into one cluster, but they are a minority there |
| earlier | `disappeared` | Its funds scattered or left clusters |
| later | `continued` | One earlier cluster makes up a piece of it |
| later | `merged` | Two or more earlier clusters each make up a piece of it |
| later | `new` | No earlier cluster does: built from funds outside a cluster or new funds |

In [ ]:
# Re-import edited fund_clustering modules automatically before each cell runs.
%load_ext autoreload
%autoreload 2

In [ ]:
# Weeks to follow, in order. Each must be run first, one week at a time:
#   python main.py --week 2026-W28   (reads data/, writes output/)
from pathlib import Path

from fund_clustering.migration import load_method_week

OUTPUT_DIR = Path("../output")  # main.py output for every week below
METHOD_ID = "hellinger"
WEEKS = [
    "2026-W28", "2026-W29", "2026-W30", "2026-W31", "2026-W32", "2026-W33",
    "2026-W34", "2026-W35", "2026-W36", "2026-W37", "2026-W38", "2026-W39",
    "2026-W40", "2026-W41", "2026-W42", "2026-W43", "2026-W44", "2026-W45",
]

# Load each week's result; compare every week with the next one.
weeks = {week: load_method_week(OUTPUT_DIR, METHOD_ID, week) for week in WEEKS}
pairs = list(zip(WEEKS, WEEKS[1:]))

# (weeks loaded, week pairs)
len(weeks), len(pairs)

In [ ]:
# Transitions for every consecutive pair, plus cluster names and sizes per week.
import pandas as pd

from fund_clustering.cluster import NOT_IN_CLUSTER
from fund_clustering.migration import cluster_transitions

# Name and fund count per cluster id, per week.
names = {w: weeks[w].points.groupby("cluster")["cluster_name"].first() for w in WEEKS}
sizes = {w: weeks[w].points["cluster"].value_counts() for w in WEEKS}

# (earlier week, later week) -> one row per cluster with its transition.
transitions = {
    (a, b): cluster_transitions(weeks[a].points["cluster"], weeks[b].points["cluster"])
    for a, b in pairs
}


def week_row(a: str, b: str) -> dict:
    # Counts per transition: what happened to week a's clusters, and how week b's arose.
    t = transitions[(a, b)]
    counts = t.groupby(["side", "transition"]).size()
    labels = weeks[b].points["cluster"]
    return {
        "week": b,
        "funds": len(labels),
        "clusters": labels[labels != NOT_IN_CLUSTER].nunique(),
        **{k: int(counts.get(("before", k), 0)) for k in
           ["survived", "split", "absorbed", "disappeared"]},
        **{k: int(counts.get(("after", k), 0)) for k in ["merged", "new"]},
    }


# One row per week: what happened since the week before.
overview = pd.DataFrame([week_row(a, b) for a, b in pairs]).set_index("week")
overview

In [ ]:
# Helper: cluster name and size for tables.
import plotly.express as px
import plotly.graph_objects as go


def label(week: str, cluster: int) -> str:
    # e.g. "Sverige · A (17)"
    if cluster == NOT_IN_CLUSTER:
        return "Not in a cluster"
    return f"{names[week][cluster]} ({sizes[week][cluster]})"

## Each week

One map per week, like in `explore.ipynb`: a dot per fund, coloured by cluster, over its cluster's territory. Made to be stitched into an animation, so:

- **Colours and names follow the cluster.** A cluster keeps the colour and name of the cluster it continues, matched by shared funds (Jaccard ≥ 0.3), also across a gap of up to `TRACK_MEMORY` weeks. Only a cluster with no predecessor gets a new colour.
- **Clusters stay in place.** The maps here are not main.py's: one 2D layout is fitted for all weeks together (AlignedUMAP, linked by ISIN), which pulls each fund towards its spot in the week before and after. A fund that really changed still moves. Territories are redrawn on these positions; all maps share one axis range. Fitting takes a few minutes.
- **Saving frames:** the camera icon on each map downloads `hellinger_<week>.png` at `FRAME_WIDTH` × `FRAME_HEIGHT`.

In [ ]:
# Cluster tracks: a cluster continues the track it shares the most funds with.
import numpy as np
from scipy.optimize import linear_sum_assignment

from fund_clustering.migration import MIN_MATCH

TRACK_MEMORY = 4  # weeks a track may be missing and still be picked up again
PALETTE = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
NOT_IN_CLUSTER_COLOUR = "#dadce0"

track_of: dict[tuple[str, int], int] = {}  # (week, cluster) -> track
track_name: dict[int, str] = {}  # name from the week the track started
track_members: dict[int, set[str]] = {}  # funds in its most recent week
track_seen: dict[int, int] = {}  # index in WEEKS of its most recent week
track_colour: dict[int, str] = {}  # a new track takes a colour no live track uses

for i, week in enumerate(WEEKS):
    labels = weeks[week].points["cluster"]
    clusters = sorted(c for c in labels.unique() if c != NOT_IN_CLUSTER)
    members = {c: set(labels.index[labels == c]) for c in clusters}
    # Tracks seen recently enough to be continued this week.
    candidates = [t for t, seen in track_seen.items() if i - seen <= TRACK_MEMORY]

    # Pair this week's clusters with tracks one to one, by Jaccard overlap of funds.
    matched = {}
    if candidates:
        # Jaccard matrix, one row per cluster this week, one column per candidate track:
        # funds in both (&) / funds in either (|), 0 = nothing shared, 1 = same funds.
        # Dividing by the union stops a big track from matching every small cluster
        # that happens to sit inside it.
        overlap = np.array([
            [len(members[c] & track_members[t]) / len(members[c] | track_members[t])
             for t in candidates]
            for c in clusters
        ])
        # linear_sum_assignment picks pairs (row, column) so that each cluster gets at
        # most one track and each track at most one cluster, with the highest total
        # overlap (the Hungarian algorithm). It returns two arrays, rows and columns;
        # zip(*...) walks them as pairs. Picking the best track per cluster on its own
        # could hand one track to both halves of a split; this can't.
        # Pairs below MIN_MATCH share too little to be the same cluster: those
        # clusters start a new track below.
        for row, column in zip(*linear_sum_assignment(overlap, maximize=True)):
            if overlap[row, column] >= MIN_MATCH:
                matched[clusters[row]] = candidates[column]

    # Unmatched clusters start a new track (new id, name and colour).
    for c in clusters:
        track = matched.get(c, len(track_name))
        if track not in track_name:
            track_name[track] = names[week][c]
            # Avoid colours of tracks that are live or were just assigned this week.
            live = candidates + [track_of[(week, d)] for d in clusters if (week, d) in track_of]
            used = {track_colour[t] for t in live}
            free = [colour for colour in PALETTE if colour not in used]
            track_colour[track] = free[0] if free else PALETTE[track % len(PALETTE)]
        # Remember the latest funds, so the next weeks match against these.
        track_of[(week, c)] = track
        track_members[track] = members[c]
        track_seen[track] = i

# (tracks over all weeks, clusters per week on average)
len(track_name), round(len(track_of) / len(WEEKS), 1)

In [ ]:
# One 2D layout for all weeks at once (AlignedUMAP), so a cluster stays where it was.
# Only the drawing positions change; clusters, names and colours still come from main.py.
import warnings

from umap import AlignedUMAP

from fund_clustering.cluster import scale_to_unit_square
from fund_clustering.load import load_week
from fund_clustering.methods import METHODS
from fund_clustering.territory import territory_grid, territory_polygons

DATA_DIR = Path("../data")  # weekly CSVs: features for the layout, names for the hover
ALIGNMENT = 0.01  # how hard a fund is pulled to last/next week's spot; higher = calmer, hides real moves

# Hellinger features per week, one row per fund in the CSV's order.
data = {week: load_week(DATA_DIR, week) for week in WEEKS}
features = [METHODS[METHOD_ID].features(data[week]) for week in WEEKS]
isins = [data[week].funds["isin"].tolist() for week in WEEKS]

# relations[k]: row of a fund in week k -> its row in week k + 1 (same ISIN).
relations = []
for earlier, later in zip(isins, isins[1:]):
    row_in_later = {isin: row for row, isin in enumerate(later)}
    relations.append(
        {row: row_in_later[isin] for row, isin in enumerate(earlier) if isin in row_in_later}
    )

# Same 2D settings as main.py's map (cluster.map_coordinates); takes a few minutes.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)  # umap: random_state disables parallelism
    layout = AlignedUMAP(
        n_neighbors=40, min_dist=0.9, spread=1.2, random_state=42,
        alignment_regularisation=ALIGNMENT,
    ).fit(features, relations=relations)

# Scale all weeks together (not one by one), so the shared positions survive.
joint = scale_to_unit_square(np.vstack(layout.embeddings_))
starts = np.cumsum([0] + [len(week_isins) for week_isins in isins])
aligned_xy = {
    week: pd.DataFrame(joint[starts[k] : starts[k + 1]], index=isins[k], columns=["x", "y"])
    for k, week in enumerate(WEEKS)
}

# Territories and label spots on the new positions, the same way main.py does it.
territories = {}  # week -> {cluster id: shape}
label_xy = {}  # week -> {cluster id: (x, y)}, median of the cluster's dots
for week in WEEKS:
    xy = aligned_xy[week].to_numpy()
    labels = weeks[week].points.loc[aligned_xy[week].index, "cluster"].to_numpy()
    territories[week] = territory_polygons(territory_grid(xy, labels))
    label_xy[week] = {
        c: tuple(np.median(xy[labels == c], axis=0))
        for c in np.unique(labels) if c != NOT_IN_CLUSTER
    }

# One axis range for all maps, so frames line up.
everything = pd.concat(aligned_xy.values())
padding = 0.03
x_range = [everything["x"].min() - padding, everything["x"].max() + padding]
y_range = [everything["y"].min() - padding, everything["y"].max() + padding]

# Median distance a fund moves from one week to the next (0..1 map).
moves = [
    aligned_xy[a].loc[shared].sub(aligned_xy[b].loc[shared]).pow(2).sum(axis=1).pow(0.5).median()
    for a, b in pairs
    for shared in [aligned_xy[a].index.intersection(aligned_xy[b].index)]
]
pd.Series(moves, index=[b for _, b in pairs], name="median move").round(3)

In [ ]:
# Map per week: dots and territories in the track colour, track names on top.
FRAME_WIDTH, FRAME_HEIGHT = 1600, 1200


def with_alpha(hex_colour: str, alpha: float) -> str:
    # "#rrggbb" -> "rgba(r,g,b,alpha)", for see-through territories.
    r, g, b = (int(hex_colour[i : i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r},{g},{b},{alpha})"


def colour_of(week: str, cluster: int) -> str:
    # The cluster's track colour; light grey outside a cluster.
    if cluster == NOT_IN_CLUSTER:
        return NOT_IN_CLUSTER_COLOUR
    return track_colour[track_of[(week, cluster)]]


def week_map(week: str) -> go.Figure:
    # Fund table: cluster, aligned position, name and category for the hover.
    display_data = data[week].funds.set_index("isin")[["name", "category"]]
    funds = weeks[week].points[["cluster", "cluster_name"]].join(aligned_xy[week])
    funds = funds.join(display_data)
    funds["colour"] = [colour_of(week, c) for c in funds["cluster"]]

    fig = go.Figure()

    # Territories (outer edges only) first, so they sit behind the dots.
    for cluster, shape in territories[week].items():
        colour = colour_of(week, cluster)
        for polygon in getattr(shape, "geoms", [shape]):  # one shape may have several parts
            x, y = polygon.exterior.xy
            fig.add_trace(go.Scatter(
                x=list(x), y=list(y), fill="toself", mode="lines",
                fillcolor=with_alpha(colour, 0.22),
                line=dict(color=with_alpha(colour, 0.6), width=0.5),
                hoverinfo="skip",
            ))

    # One dot per fund, in its cluster's colour.
    fig.add_trace(go.Scatter(
        x=funds["x"], y=funds["y"], mode="markers",
        marker=dict(color=funds["colour"], size=6),
        customdata=funds.reset_index()[["isin", "category", "cluster_name"]],
        text=funds["name"],
        hovertemplate=(
            "<b>%{text}</b><br>%{customdata[0]}<br>%{customdata[1]}"
            "<br>%{customdata[2]}<extra></extra>"
        ),
    ))

    # Track names at each cluster's label position.
    for cluster, (x, y) in label_xy[week].items():
        fig.add_annotation(
            x=x, y=y, showarrow=False,
            text=track_name[track_of[(week, cluster)]],
            font=dict(size=10, color="#202124"),
            bgcolor="rgba(255,255,255,0.6)",
        )

    # Fixed size, white background, no axes: every frame looks the same.
    clustered = funds["cluster"] != NOT_IN_CLUSTER
    fig.update_layout(
        title=f"Hellinger 60/40 · {week} · {funds.loc[clustered, 'cluster'].nunique()} clusters",
        showlegend=False, width=FRAME_WIDTH, height=FRAME_HEIGHT,
        plot_bgcolor="white",
    )
    fig.update_xaxes(range=x_range, visible=False)
    fig.update_yaxes(range=y_range, visible=False, scaleanchor="x")
    return fig


def show_week(week: str) -> None:
    # Draw one week's map; the camera button saves it as <method>_<week>.png.
    week_map(week).show(config={"toImageButtonOptions": {
        "format": "png", "filename": f"{METHOD_ID}_{week}",
        "width": FRAME_WIDTH, "height": FRAME_HEIGHT,
    }})

In [ ]:
# Map 2026-W28
show_week("2026-W28")

In [ ]:
# Map 2026-W29
show_week("2026-W29")

In [ ]:
# Map 2026-W30
show_week("2026-W30")

In [ ]:
# Map 2026-W31
show_week("2026-W31")

In [ ]:
# Map 2026-W32
show_week("2026-W32")

In [ ]:
# Map 2026-W33
show_week("2026-W33")

In [ ]:
# Map 2026-W34
show_week("2026-W34")

In [ ]:
# Map 2026-W35
show_week("2026-W35")

In [ ]:
# Map 2026-W36
show_week("2026-W36")

In [ ]:
# Map 2026-W37
show_week("2026-W37")

In [ ]:
# Map 2026-W38
show_week("2026-W38")

In [ ]:
# Map 2026-W39
show_week("2026-W39")

In [ ]:
# Map 2026-W40
show_week("2026-W40")

In [ ]:
# Map 2026-W41
show_week("2026-W41")

In [ ]:
# Map 2026-W42
show_week("2026-W42")

In [ ]:
# Map 2026-W43
show_week("2026-W43")

In [ ]:
# Map 2026-W44
show_week("2026-W44")

In [ ]:
# Map 2026-W45
show_week("2026-W45")

## Splits, merges and the rest

Every cluster that did not simply survive or continue, week by week, with where its funds went (or came from) and the shares.

In [ ]:
# One row per event. Partner shares are of this cluster's funds.
def event_rows(a: str, b: str) -> list[dict]:
    rows = []
    for t in transitions[(a, b)].itertuples():
        if t.transition in ("survived", "continued"):
            continue  # nothing happened
        # Partners live in the other week of the pair.
        week, other = (a, b) if t.side == "before" else (b, a)
        rows.append({
            "week": b,
            "transition": t.transition,
            "cluster": label(week, t.cluster),
            "partners": "; ".join(
                f"{label(other, p)} {s:.0%}" for p, s in zip(t.partners, t.shares)
            ),
        })
    return rows


# All events over all week pairs.
events = pd.DataFrame([row for a, b in pairs for row in event_rows(a, b)])
events.set_index(["week", "transition"])

## Example funds

The five funds from `explore.ipynb`, plus three picked for illustrations: which cluster each sat in every week. When the cluster did not simply survive into the next week, the cell also says what happened (e.g. `→ split`). Empty = the fund is not in that week's data.

| ISIN | Name |
|------|------|
| `LU0256331488` | Schroder ISF Global Energy A Acc USD |
| `NO0013528349` | DNB European Defence S |
| `LU0069152568` | UBS (Lux) EF Biotech (USD) P-acc |
| `LU0256846568` | Pictet-Security R USD |
| `SE0015837778` | Go Blockchain Fund A |
| `LU0194779913` | Invesco Pan Eurp Cr Systmtc Eq A EUR Acc |
| `LU1777968246` | Kavaljer Investmentbolagsfond A |
| `LU0354788688` | CB Save Earth Fund RC |

**Why the last three.** On the aligned maps the median fund's biggest weekly jump is 0.03 (map is 0..1), so the first five change colour but hardly move. These three show the two kinds of change:

- **Real move, `LU0194779913` (Invesco):** jumps 0.35 in W42. Its category was "Europa, Mix bolag" all along, but until W41 the data showed a global portfolio (USA 41%, Japan 17%). From W42 it is European (UK 19%, France 15%, Germany 12%), and the fund lands in the 83-fund Europe cluster next to its peers (and next to DNB European Defence).
- **Real move, `LU1777968246` (Kavaljer):** the biggest jump of all, 0.61 in W32: Sweden 63% → 46%, USA 21% → 33%. Both before and after, its cluster is named "Global & Sverige", so the move shows on the map but not in the name.
- **Noise, `LU0354788688` (CB Save Earth):** its dot barely moves (biggest jump 0.02), but its cluster changes 10 times as the environment cluster splits and re-merges.

Together: one dot travels because the fund changed; another stays put while its colour flickers because the clustering changed.

In [ ]:
# Example funds (see explore.ipynb); try any ISIN here.
EXAMPLE_ISINS = [
    "LU0256331488",  # Schroder ISF Global Energy A Acc USD
    "NO0013528349",  # DNB European Defence S
    "LU0069152568",  # UBS (Lux) EF Biotech (USD) P-acc
    "LU0256846568",  # Pictet-Security R USD
    "SE0015837778",  # Go Blockchain Fund A
    "LU0194779913",  # Invesco Pan Eurp Cr Systmtc Eq: global -> European data in W42
    "LU1777968246",  # Kavaljer Investmentbolagsfond A: Sweden -> USA shift in W32
    "LU0354788688",  # CB Save Earth Fund RC: stays put, cluster flickers
]


def next_transition(week: str, cluster: int) -> str | None:
    # What happens to this cluster by the next week; None in the last week or outside a cluster.
    i = WEEKS.index(week)
    if i == len(WEEKS) - 1 or cluster == NOT_IN_CLUSTER:
        return None
    t = transitions[(week, WEEKS[i + 1])].query("side == 'before' and cluster == @cluster")
    return t["transition"].iloc[0]


def fund_cell(week: str, isin: str) -> str | None:
    # "Cluster name (size)", plus "→ split" etc. when the cluster changes next week.
    points = weeks[week].points
    if isin not in points.index:
        return None
    cluster = int(points.loc[isin, "cluster"])
    after = next_transition(week, cluster)
    return label(week, cluster) + (f" → {after}" if after not in (None, "survived") else "")


# One row per week, one column per fund (named by the latest name in the data).
fund_names = pd.concat(d.funds.set_index("isin")["name"] for d in data.values())
fund_names = fund_names[~fund_names.index.duplicated(keep="last")]
pd.DataFrame(
    {fund_names.get(isin, isin): [fund_cell(w, isin) for w in WEEKS] for isin in EXAMPLE_ISINS},
    index=pd.Index(WEEKS, name="week"),
)

## Summary: 2026-W28 → 2026-W45, Hellinger

- **Funds are steady, cluster counts are not.** 768–789 funds every week, but 35 to 49 clusters.
- **Most clusters survive a week**: 23–38 of them, depending on the week. Real splits are rare, 0–4 per week.
- **The common event is `absorbed`, then `merged` back.** In W39, 21 small clusters were folded into three big ones (`Sverige` 64 funds, a mixed environment/new-energy cluster of 48, a mixed US cluster of 58). W43 had 18 absorptions and W36/W37 had 13 each. Most of those groups come back a week or two later.
- **The environment funds flip-flop rather than split.** `LU0354788688`, `SE0017767577`, `LU2553958583` and the water fund `LU1890834598` sit in a 46–48-fund mixed cluster in W28–W35, W37, W39 and W43. In W36, W38, W40–W42 and W44–W45 they are in a small environment cluster of 5–9 funds instead. The W28 → W45 "split" in `cluster_migration.ipynb` is one side of that flip, not a one-off change in holdings.
- **Takeaway:** week-to-week labels are unstable for groups on a density border. Neighbours are the stable signal for search. Keeping map labels stable would need reference clusters with hysteresis.